# Proyecto ML - Prediccion de cancelaciones hoteleras

## 📖 Tema del proyecto

Mi proyecto va a tratar sobre **prediccion de cancelaciones hoteleras**.

La idea es construir un modelo de Machine Learning que ayude a estimar si una reserva sera cancelada antes de la llegada del cliente.

**Aplicacion de negocio:** anticipar cancelaciones para mejorar ocupacion, ingresos, politicas de deposito y decisiones de overbooking controlado.

## 🌐 Fuente y dataset

**Fuente:** Kaggle  
**Dataset:** Hotel Booking Demand  
**URL:** https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

Incluye informacion sobre la reserva, el cliente, el canal de venta, fechas, historial, deposito, precio y estado final de la reserva.

In [14]:
import pandas as pd

df = pd.read_csv("../Data/Raw/hotel_bookings.csv")
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 📊 Tamano del dataset

El dataset cumple los requisitos minimos del proyecto porque tiene mas de **1.000 registros** y mas de **10 campos**.

In [18]:
print("=====Dataset=====")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

=====Dataset=====
Filas: 119,390
Columnas: 32


## 🎯 Target del modelo (Y)

La variable objetivo sera `is_canceled`.

- `0`: reserva no cancelada.
- `1`: reserva cancelada.

In [12]:
target_table = pd.DataFrame({
    "reservas": df["is_canceled"].value_counts().sort_index(),
    "porcentaje": (df["is_canceled"].value_counts(normalize=True).sort_index() * 100).round(1),
})
target_table

,reservas,porcentaje
is_canceled,,
0,75166,63.0
1,44224,37.0


## 📈 Variables predictoras candidatas (X)

Estas son las variables que usare inicialmente como features o variables predictoras.

| Grupo | Variables |
|---|---|
| Tipo de hotel | `hotel` |
| Antelacion y fechas | `lead_time`, `arrival_date_year`, `arrival_date_month`, `arrival_date_week_number`, `arrival_date_day_of_month` |
| Duracion de estancia | `stays_in_weekend_nights`, `stays_in_week_nights` |
| Personas en la reserva | `adults`, `children`, `babies` |
| Regimen y pais | `meal`, `country` |
| Canal y segmento | `market_segment`, `distribution_channel`, `customer_type` |
| Historial del cliente | `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled` |
| Habitacion y cambios | `reserved_room_type`, `booking_changes` |
| Condiciones de reserva | `deposit_type`, `days_in_waiting_list`, `adr` |
| Servicios adicionales | `required_car_parking_spaces`, `total_of_special_requests` |


In [13]:
predictor_columns = [
    "hotel", "lead_time", "arrival_date_year", "arrival_date_month",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "meal", "country",
    "market_segment", "distribution_channel", "customer_type",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "reserved_room_type",
    "booking_changes", "deposit_type", "days_in_waiting_list",
    "adr", "required_car_parking_spaces", "total_of_special_requests",
]

pd.DataFrame({
    "variable_predictora": predictor_columns,
    "tipo_dato": [str(df[col].dtype) for col in predictor_columns],
    "nulos": [df[col].isna().sum() for col in predictor_columns],
})

,variable_predictora,tipo_dato,nulos
0,hotel,str,0
1,lead_time,int64,0
2,arrival_date_year,int64,0
3,arrival_date_month,str,0
4,arrival_date_week_number,int64,0
5,arrival_date_day_of_month,int64,0
6,stays_in_weekend_nights,int64,0
7,stays_in_week_nights,int64,0
8,adults,int64,0
9,children,float64,4


## ❌ ⁉️ Variables excluidas o a revisar

Estas variables no entraran directamente en el primer modelo o necesitan una revision especial.

| Variable | Decision | Motivo / tratamiento |
|---|---|---|
| `reservation_status` | Excluir | Data leakage: indica el estado final de la reserva. |
| `reservation_status_date` | Excluir | Data leakage: fecha del estado final. No estaria disponible al predecir. |
| `assigned_room_type` | Revisar / excluir al inicio | Puede depender de decisiones posteriores a la reserva inicial. Usare primero `reserved_room_type`. |
| `company` | No usar directamente | Tiene aproximadamente un 94% de valores nulos. Mejor eliminarla o crear `has_company`. |
| `agent` | Revisar | Es un identificador de agencia, no una cantidad numerica. Si se usa, tratarlo como categorica e imputar nulos. |
| `country` | Mantener, pero imputar | Tiene pocos nulos. Imputar con `Unknown` antes de modelar. |
| `children` | Mantener, pero imputar | Tiene muy pocos valores nulos. Imputar con `0` o con la mediana. |


In [ ]:
excluded_or_review = [
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "company",
    "agent",
    "country",
    "children",
]

pd.DataFrame({
    "variable": excluded_or_review,
    "tipo_dato": [str(df[col].dtype) for col in excluded_or_review],
    "nulos": [df[col].isna().sum() for col in excluded_or_review],
    "porcentaje_nulos": [(df[col].isna().mean() * 100).round(2) for col in excluded_or_review],
})

## 📉 Transformaciones previstas en ingenieria de datos

Despues de decidir que variables usar o revisar, las transformaciones principales seran:

1. Codificar variables categoricas con One-Hot Encoding.
2. Escalar variables numericas para modelos sensibles a escala, como regresion logistica, KNN o SVM.
3. Revisar outliers en variables como `lead_time`, `adr`, `stays_in_week_nights` y `booking_changes`.
4. Crear variables nuevas si aportan valor, por ejemplo:
   - `total_nights = stays_in_weekend_nights + stays_in_week_nights`
   - `total_guests = adults + children + babies`
   - `has_company`
   - `has_special_requests`
   - `needs_parking`
5. Separar correctamente `X` e `y`, usando `is_canceled` como target.
6. Dividir los datos en train/test antes de entrenar los modelos.


In [ ]:
# Ejemplo de transformaciones iniciales que se podrian aplicar despues de la presentacion.

df_engineered = df.copy()

df_engineered["children"] = df_engineered["children"].fillna(0)
df_engineered["country"] = df_engineered["country"].fillna("Unknown")
df_engineered["agent"] = df_engineered["agent"].fillna(0).astype(str)
df_engineered["has_company"] = df_engineered["company"].notna().astype(int)

df_engineered["total_nights"] = (
    df_engineered["stays_in_weekend_nights"] + df_engineered["stays_in_week_nights"]
)
df_engineered["total_guests"] = (
    df_engineered["adults"] + df_engineered["children"] + df_engineered["babies"]
)
df_engineered["has_special_requests"] = (df_engineered["total_of_special_requests"] > 0).astype(int)
df_engineered["needs_parking"] = (df_engineered["required_car_parking_spaces"] > 0).astype(int)

df_engineered[[
    "children", "country", "agent", "has_company",
    "total_nights", "total_guests", "has_special_requests", "needs_parking"
]].head()